# Inferencia §4.2 — Robustez ante clima (Swin + FlashInternImage)

Notebook de **Google Colab (GPU T4)** para correr la evaluación de robustez del TFM
sobre el dataset de clima (dry / wet / half), reutilizando `mim test` de MMSegmentation.

## Qué subir a tu Google Drive antes de empezar
Crea una carpeta `MyDrive/tfm_colab/` con esta estructura:
```
tfm_colab/
  code/                         <- sube aquí la carpeta 'code' del bundle (~/Descargas/tfm_colab/code)
    swin-T-512x512/             (config Swin)
    flashInternImage-T-512x512/ (config + custom_modules/backbone de Flash)
    ops_dcnv4/                  (fuente DCNv4 a compilar)
    weather_splits/             (dry.txt wet.txt half.txt all_test.txt)
  checkpoints/
    <tu_swin>.pth               <- pon aquí tus checkpoints entrenados
    <tu_flash>.pth
```
Los **datos de clima** NO hay que subirlos: el notebook los baja de tu Kaggle (público).

## Orden de ejecución
1. Comprobar GPU → 2. Instalar entorno (**reiniciar runtime tras esta celda**) →
3. Montar Drive + copiar código → 4. Bajar datos + splits + sanity de clases →
5. Compilar DCNv4 (para Flash) → 6. Inferencia Swin y Flash por condición.

> Aviso: es una v1. El entorno de Colab cambia; si una celda de instalación/compilación
> falla, pásame el error y la ajustamos. La lógica de inferencia/métricas es la buena.


## 1) Comprobar GPU (debe ser T4 / sm_75)

In [ ]:
!nvidia-smi -L
import torch
print("torch:", torch.__version__, "| cuda:", torch.version.cuda, "| disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| capability:", torch.cuda.get_device_capability(0))


## 2) Entorno — en DOS pasos con **reinicio en medio**
`mmcv==2.1.0` no soporta torch>2.1, así que fijamos **torch 2.1.2 + cu118**. Reinstalar torch
*y seguir instalando en la misma celda* rompe el kernel (el `[object CloseEvent]` que viste).
Hazlo así: ejecuta **2a** → **reinicia la sesión** → ejecuta **2b**. Usamos la *wheel precompilada*
de mmcv (no compila desde fuente, que es lo que se colgaba).

In [ ]:
# 2a) SOLO torch. No instales nada más en esta celda.
!pip -q install torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu118
print(">>> torch 2.1.2 instalado.  AHORA: Entorno de ejecucion > Reiniciar sesion.  Luego ejecuta la celda 2b.")


### ⚠️ Reinicia el runtime AQUÍ (*Entorno de ejecución ▸ Reiniciar sesión*) antes de ejecutar 2b.

In [ ]:
# 2b) Tras reiniciar: verifica torch y luego el stack MMSeg con WHEEL PRECOMPILADA (no compila -> no se cuelga)
import torch
print('torch', torch.__version__, '| cuda', torch.version.cuda)   # debe ser 2.1.2 / 11.8
assert torch.__version__.startswith('2.1'), 'Reinicia el runtime: sigue cargada una version vieja de torch'
!pip -q install -U openmim
!pip -q install "mmcv==2.1.0" -f https://download.openmmlab.com/mmcv/dist/cu118/torch2.1/index.html
!pip -q install "mmengine>=0.7.4" "mmdet>=3.0.0" "mmsegmentation>=1.2.2"
!pip -q install ftfy regex seaborn scikit-learn timm yacs
import mmcv, mmseg
print('mmcv', mmcv.__version__, '| mmseg', mmseg.__version__, '>>> entorno listo, sigue en la celda 3.')


## 3) Montar Drive y copiar el código a espacio de trabajo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/tfm_colab'   # <-- EDITA si lo pusiste en otra ruta
CODE     = f'{DRIVE_ROOT}/code'
CKPT_DIR = f'{DRIVE_ROOT}/checkpoints'
assert os.path.isdir(CODE), f'No existe {CODE}. Sube la carpeta code/ del bundle a tu Drive.'
print('code/       ->', sorted(os.listdir(CODE)))
print('checkpoints/->', sorted(os.listdir(CKPT_DIR)) if os.path.isdir(CKPT_DIR) else 'NO existe (sube tus .pth)')


In [ ]:
import os, shutil
WORK = '/content/work'; os.makedirs(WORK, exist_ok=True)
for d in ['swin-T-512x512','flashInternImage-T-512x512','ops_dcnv4','weather_splits']:
    src, dst = f'{CODE}/{d}', f'{WORK}/{d}'
    assert os.path.isdir(src), f'Falta {src} en Drive'
    if os.path.exists(dst): shutil.rmtree(dst)
    shutil.copytree(src, dst)
print('copiado a', WORK, '->', sorted(os.listdir(WORK)))


## 4) Bajar datos de clima (Kaggle público) + splits + sanity de clases

In [ ]:
%cd /content
!curl -sL -o weather.zip "https://www.kaggle.com/api/v1/datasets/download/lauraparejaprieto/weather-adverse-road-defect-semantic-segmentation"
!unzip -q -o weather.zip -d /content
import os
DATA = '/content/final_dataset'
print('contenido:', sorted(os.listdir(DATA)))
print('nº imágenes:', len(os.listdir(f'{DATA}/images')), '| nº labels:', len(os.listdir(f'{DATA}/labels')))


In [ ]:
import os, shutil, glob, numpy as np
from PIL import Image
DATA = '/content/final_dataset'
os.makedirs(f'{DATA}/splits', exist_ok=True)
for f in ['all_test.txt','dry.txt','wet.txt','half.txt']:
    shutil.copy(f'/content/work/weather_splits/{f}', f'{DATA}/splits/{f}')
# el config usa por defecto splits/test.txt -> lo apuntamos a todo el test de clima
shutil.copy(f'{DATA}/splits/all_test.txt', f'{DATA}/splits/test.txt')
for f in ['all_test','dry','wet','half']:
    n = sum(1 for _ in open(f'{DATA}/splits/{f}.txt'))
    print(f'  split {f:9s}: {n} imgs')

# SANITY: las máscaras de clima deben usar el MISMO esquema 0..8 (+255 ignore) que el entrenamiento
CLASSES = ["bg","cracks","cracks_alligator","cracks_severe","edge_cracks","fretting","pothole","manhole","pole_shadow"]
vals = set()
for p in glob.glob(f'{DATA}/labels/*.png')[:40]:
    vals |= set(np.unique(np.array(Image.open(p))).tolist())
print('valores de clase encontrados en labels:', sorted(vals))
print('esperado: subconjunto de', list(range(len(CLASSES))), '(+255 = ignore)')
assert vals <= set(range(len(CLASSES))) | {255}, 'OJO: las máscaras de clima NO usan el esquema 0..8 -> revisar mapeo antes de medir mIoU'
print('OK: esquema de clases compatible.')


## 5) Compilar DCNv4 para T4 (sm_75) — necesario solo para FlashInternImage
Swin no lo necesita. Si esta celda falla, pásame el log; hay un fallback `core_op='DCNv4_pytorch'` (más lento).

In [ ]:
import os, sys
os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'   # Tesla T4
os.environ['FORCE_CUDA'] = '1'
%cd /content/work/ops_dcnv4
!python setup.py build install 2>&1 | tail -25
%cd /content
# verificar import del op compilado
try:
    import DCNv4  # noqa
    print('\n>>> DCNv4 import OK')
except Exception as e:
    print('\n>>> DCNv4 import FALLO:', repr(e))
    print('    (si falla, en la celda de Flash pon core_op=DCNv4_pytorch via --cfg-options)')


## 6) Inferencia con `mim test` por condición
`run_test` corre desde la carpeta del modelo (para que `custom_imports`/`custom_modules` resuelvan)
y sobre-escribe el dataset para apuntar al dataset de clima (`seg_map_path=labels`, `ann_file=splits/<cond>.txt`).
Imprime la tabla de MMSegmentation con **aAcc / mIoU / IoU por clase** de esa condición.

In [ ]:
import subprocess, os, re

def run_test(name, model_dir, ckpt, cond, seg_dir='labels', data_root='/content/final_dataset', extra=None):
    assert os.path.isfile(ckpt), f'No existe el checkpoint: {ckpt}'
    workdir = f'/content/out/{name}_{cond}'; os.makedirs(workdir, exist_ok=True)
    cfg_opts = [
        f'test_dataloader.dataset.data_root={data_root}',
        f'test_dataloader.dataset.ann_file=splits/{cond}.txt',
        f'test_dataloader.dataset.data_prefix.seg_map_path={seg_dir}',
    ] + (extra or [])
    cmd = ['mim','test','mmseg','config.py','--checkpoint',ckpt,'--work-dir',workdir,'--cfg-options',*cfg_opts]
    env = dict(os.environ, PYTHONPATH=model_dir)
    print(f'\n{"="*70}\n[{name} | {cond}]  ckpt={os.path.basename(ckpt)}\n{"="*70}')
    p = subprocess.run(cmd, cwd=model_dir, env=env, capture_output=True, text=True)
    out = p.stdout + p.stderr
    # imprime la parte final (tabla de métricas)
    print(out[-2500:])
    m = re.search(r'mIoU[\s|:]+([0-9.]+)', out)
    a = re.search(r'aAcc[\s|:]+([0-9.]+)', out)
    return {'name':name,'cond':cond,'mIoU':(m.group(1) if m else '?'),'aAcc':(a.group(1) if a else '?')}


### 6a) Swin — todas las condiciones

In [ ]:
SWIN_DIR  = '/content/work/swin-T-512x512'
SWIN_CKPT = f'{CKPT_DIR}/swin.pth'   # <-- EDITA al nombre real de tu checkpoint de Swin en Drive
res = []
for cond in ['all_test','dry','wet','half']:
    res.append(run_test('swin', SWIN_DIR, SWIN_CKPT, cond))
import pandas as pd; display(pd.DataFrame(res))


### 6b) FlashInternImage — todas las condiciones

In [ ]:
FLASH_DIR  = '/content/work/flashInternImage-T-512x512'
FLASH_CKPT = f'{CKPT_DIR}/flash.pth'  # <-- EDITA al nombre real de tu checkpoint de Flash en Drive
# si DCNv4 compilado falla, descomenta la linea 'extra' para usar el fallback en pytorch:
extra = None  # extra = ['model.backbone.core_op=DCNv4_pytorch']
res_f = []
for cond in ['all_test','dry','wet','half']:
    res_f.append(run_test('flash', FLASH_DIR, FLASH_CKPT, cond, extra=extra))
import pandas as pd; display(pd.DataFrame(res_f))


### 6c) Resumen §4.2 — mIoU por backbone y condición

In [ ]:
import pandas as pd
tabla = pd.DataFrame(res + res_f).pivot(index='name', columns='cond', values='mIoU')
tabla = tabla[['all_test','dry','wet','half']]
print('mIoU (%) por condición climática:')
display(tabla)
tabla.to_csv('/content/out/summary_weather_miou.csv')
print('guardado en /content/out/summary_weather_miou.csv')


## 7) (Opcional) Matriz de confusión por píxel — estilo `force_predictions.py`
Reutiliza `init_model` + `inference_model`. Cambia `MODEL_DIR`/`CKPT` para swin o flash.

In [ ]:
import os, numpy as np, mmengine, torch
from mmseg.apis import init_model, inference_model
from mmseg.registry import DATASETS
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt, seaborn as sns

MODEL_DIR = '/content/work/swin-T-512x512'   # o flashInternImage-T-512x512
CKPT      = SWIN_CKPT                          # o FLASH_CKPT
COND      = 'all_test'
CLASSES = ["bg","cracks","cracks_alligator","cracks_severe","edge_cracks","fretting","pothole","manhole","pole_shadow"]

os.chdir(MODEL_DIR); import sys; sys.path.insert(0, MODEL_DIR)
cfg = mmengine.Config.fromfile('config.py')
cfg.test_dataloader.dataset.data_root = '/content/final_dataset'
cfg.test_dataloader.dataset.ann_file = f'splits/{COND}.txt'
cfg.test_dataloader.dataset.data_prefix.seg_map_path = 'labels'
model = init_model(cfg, CKPT, device='cuda:0')
ds = DATASETS.build(cfg.test_dataloader.dataset)

cm = np.zeros((9,9), dtype=np.int64)
with torch.no_grad():
    for i in range(len(ds)):
        s = ds[i]; gt = s['data_samples'].gt_sem_seg.data[0].cpu().numpy().flatten()
        pred = inference_model(model, s['data_samples'].img_path).pred_sem_seg.data[0].cpu().numpy().flatten()
        mask = (gt>=0)&(gt<9)
        if mask.any(): cm += confusion_matrix(gt[mask], pred[mask], labels=range(9))
        if (i+1)%50==0: print(f'{i+1}/{len(ds)}')
rs = cm.sum(1, keepdims=True); cmn = np.divide(cm.astype(float), rs, out=np.zeros_like(cm,float), where=rs!=0)
plt.figure(figsize=(11,9)); sns.heatmap(cmn, annot=True, fmt='.2f', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES)
plt.title(f'Confusion (recall norm) — {COND}'); plt.ylabel('GT'); plt.xlabel('Pred'); plt.tight_layout()
os.makedirs('/content/out', exist_ok=True); plt.savefig('/content/out/cm.png', bbox_inches='tight'); plt.show()
